In [5]:
!pip install transformers datasets trl accelerate scikit-learn -q

In [6]:
# ==============================================================
# Imports
# ==============================================================

import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from sklearn.metrics import confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)


In [7]:
# ==============================================================
# GLOBALS
# ==============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 256

In [ ]:
from datasets import load_dataset

# ==============================================================
# LOAD HUMAN-LIKE DPO DATASET
# ==============================================================

# Only the train split exists
dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")

# Create train / validation split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_pref = dataset["train"].select(range(2000))
val_pref = dataset["test"].select(range(500))

In [11]:
# ==============================================================
# BASE MODEL FOR SFT (GROUNDED ON CHOSEN TEXT)
# ==============================================================

base_model_name = "gpt2"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_tokenizer.pad_token = base_tokenizer.eos_token

base_tokenizer.padding_side = "right"

# Use a CausalLM LM instead of classification head
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
base_model.to(device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3878.64it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [14]:
# ==============================================================
# SFT DATA PREPARATION
# ==============================================================

def sft_anthropic_tokenize(example):
    # Build a single full text: prompt + chosen completion
    text = example["prompt"] + example["chosen"]
    outputs = base_tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    # For causal modeling, labels == input_ids
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

sft_train = train_pref.map(
    sft_anthropic_tokenize,
    remove_columns=["prompt", "chosen", "rejected"],
)

sft_val = val_pref.map(
    sft_anthropic_tokenize,
    remove_columns=["prompt", "chosen", "rejected"],
)

sft_train.set_format(type="torch")
sft_val.set_format(type="torch")

sft_collator = DataCollatorWithPadding(base_tokenizer)

sft_train_loader = DataLoader(
    sft_train,
    batch_size=16,
    shuffle=True,
    collate_fn=sft_collator,
)

sft_val_loader = DataLoader(
    sft_val,
    batch_size=16,
    shuffle=False,
    collate_fn=sft_collator,
)

Map: 100%|██████████| 500/500 [00:00<00:00, 1189.50 examples/s]


In [15]:
# ==============================================================
# SFT TRAINING LOOP
# ==============================================================

sft_optimizer = torch.optim.AdamW(base_model.parameters(), lr=2e-5)

# LR schedule
total_sft_steps = len(sft_train_loader) * 3
sft_scheduler = get_linear_schedule_with_warmup(
    sft_optimizer,
    num_warmup_steps=int(0.1 * total_sft_steps),
    num_training_steps=total_sft_steps,
)

def train_sft_epoch():
    base_model.train()
    total_loss = 0

    for batch in sft_train_loader:
        batch = batch.to(device)

        outputs = base_model(**batch)
        loss = outputs.loss

        sft_optimizer.zero_grad()
        loss.backward()
        sft_optimizer.step()
        sft_scheduler.step()

        total_loss += loss.item()

    return total_loss / len(sft_train_loader)

def validate_sft_epoch():
    base_model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in sft_val_loader:
            batch = batch.to(device)
            outputs = base_model(**batch)
            total_loss += outputs.loss.item()

    return total_loss / len(sft_val_loader)

print("=== SFT TRAINING ===")

for epoch in range(3):
    train_loss = train_sft_epoch()
    val_loss = validate_sft_epoch()

    print(f"Epoch {epoch} — Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Save SFT model
os.makedirs("anthropic_sft_model", exist_ok=True)
base_model.save_pretrained("anthropic_sft_model")
base_tokenizer.save_pretrained("anthropic_sft_model")

=== SFT TRAINING ===


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 0 — Train Loss: 2.3258, Val Loss: 1.8470
Epoch 1 — Train Loss: 1.8796, Val Loss: 1.7674
Epoch 2 — Train Loss: 1.8163, Val Loss: 1.7477


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.85s/it]


('anthropic_sft_model/tokenizer_config.json',
 'anthropic_sft_model/tokenizer.json')

In [16]:
# ==============================================================
# REWARD MODEL DATA PREPARATION
# ==============================================================

# We'll build a dataset with
# input_ids_chosen, attention_mask_chosen, input_ids_rejected, attention_mask_rejected

def reward_anthropic_tokenize(example):
    chosen_text = example["prompt"] + example["chosen"]
    rejected_text = example["prompt"] + example["rejected"]

    chosen_enc = base_tokenizer(
        chosen_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    rejected_enc = base_tokenizer(
        rejected_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

    return {
        "input_ids_chosen": chosen_enc["input_ids"],
        "attention_mask_chosen": chosen_enc["attention_mask"],
        "input_ids_rejected": rejected_enc["input_ids"],
        "attention_mask_rejected": rejected_enc["attention_mask"],
    }

reward_train = train_pref.map(
    reward_anthropic_tokenize,
    remove_columns=["prompt", "chosen", "rejected"],
)

reward_val = val_pref.map(
    reward_anthropic_tokenize,
    remove_columns=["prompt", "chosen", "rejected"],
)

reward_train.set_format(type="torch")
reward_val.set_format(type="torch")

reward_loader = DataLoader(reward_train, batch_size=32, shuffle=True)
reward_val_loader = DataLoader(reward_val, batch_size=32, shuffle=False)


Map: 100%|██████████| 500/500 [00:00<00:00, 737.77 examples/s]


In [17]:
# ==============================================================
# REWARD MODEL ARCHITECTURE
# ==============================================================

class RewardHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, hidden):
        return self.linear(hidden)

class PairwiseRewardModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.reward = RewardHead(self.base.config.n_embd)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        hidden = outputs.hidden_states[-1]
        # reward per token
        return self.reward(hidden).squeeze(-1)

loaded_base = AutoModelForCausalLM.from_pretrained("anthropic_sft_model")
loaded_base.to(device)

reward_model = PairwiseRewardModel(loaded_base).to(device)

reward_optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-4)
bce_loss = nn.BCEWithLogitsLoss()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4061.14it/s]


In [18]:
# ==============================================================
# REWARD TRAINING LOOP
# ==============================================================

def train_reward_epoch():
    reward_model.train()
    total_loss = 0

    for batch in reward_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        chosen_scores = reward_model(
            input_ids=batch["input_ids_chosen"],
            attention_mask=batch["attention_mask_chosen"],
        )
        rejected_scores = reward_model(
            input_ids=batch["input_ids_rejected"],
            attention_mask=batch["attention_mask_rejected"],
        )

        target = torch.ones_like(chosen_scores)
        loss = bce_loss(chosen_scores - rejected_scores, target)

        reward_optimizer.zero_grad()
        loss.backward()
        reward_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(reward_loader)

def validate_reward_epoch():
    reward_model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in reward_val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            chosen_scores = reward_model(
                input_ids=batch["input_ids_chosen"],
                attention_mask=batch["attention_mask_chosen"],
            )
            rejected_scores = reward_model(
                input_ids=batch["input_ids_rejected"],
                attention_mask=batch["attention_mask_rejected"],
            )

            target = torch.ones_like(chosen_scores)
            loss = bce_loss(chosen_scores - rejected_scores, target)

            total_loss += loss.item()

    return total_loss / len(reward_val_loader)

print("=== REWARD MODEL TRAINING ===")

for epoch in range(3):
    tr_loss = train_reward_epoch()
    val_loss = validate_reward_epoch()
    print(f"Reward Epoch {epoch} — Train: {tr_loss:.4f}, Val: {val_loss:.4f}")

# Save reward model
torch.save(reward_model.state_dict(), "anthropic_reward.pt")

=== REWARD MODEL TRAINING ===
Reward Epoch 0 — Train: 0.1486, Val: 0.0530
Reward Epoch 1 — Train: 0.0538, Val: 0.0524
Reward Epoch 2 — Train: 0.0514, Val: 0.0522


In [19]:
# ==============================================================
# QUICK EVALUATION
# ==============================================================

reward_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in reward_val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        chosen_scores = reward_model(
            input_ids=batch["input_ids_chosen"],
            attention_mask=batch["attention_mask_chosen"],
        )
        rejected_scores = reward_model(
            input_ids=batch["input_ids_rejected"],
            attention_mask=batch["attention_mask_rejected"],
        )

        correct += (chosen_scores > rejected_scores).sum().item()
        total += chosen_scores.numel()

print("Reward Accuracy:", correct / total)

Reward Accuracy: 0.92503125


In [20]:
# %%
import torch
from torch import nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 256

In [21]:
# %%
# --- REUSE: Base model and tokenizer from previous PPO/SFT setup ---
base_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

policy_model = AutoModelForCausalLM.from_pretrained(base_model_name).to(device)
# Frozen reference model
ref_model = AutoModelForCausalLM.from_pretrained(base_model_name).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3554.37it/s]


In [23]:
# Only the train split exists
dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")

# Create train / validation split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_pref = dataset["train"].select(range(2000))
val_pref = dataset["test"].select(range(500))

In [25]:
print(train_pref[0])

{'prompt': 'Have you ever tried cooking a new cuisine from scratch? How did it turn out?', 'chosen': 'You know, I\'ve attempted to cook a new cuisine from scratch a few times, and let\'s just say it\'s been an... adventure 🤣. I think the most memorable one was when I tried to make Korean bibimbap from scratch.\n\nI followed the recipe to the letter, but things started to go downhill when I realized I didn\'t have the right type of Korean chili flakes (gochugaru). I substituted it with some other chili powder, thinking it would be close enough... big mistake! The flavor was way off, and it ended up being super spicy ☀️.\n\nBut the real kicker was when I tried to make the fried egg on top. I\'ve never been great at frying eggs, and this time was no exception. It ended up being all soggy and overcooked 😂. I mean, who puts a soggy egg on top of a beautiful bibimbap? This guy, apparently 🙋\u200d♂️.\n\nDespite the mishaps, my friends were super supportive and helped me devour the, uh, "inter

In [26]:
# %%
# --- Reuse: DPO tokenization adapted from previous reward model ---
def tokenize_pair(prompt, response):
    enc = tokenizer(
        prompt + response,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    return enc.input_ids.squeeze(0), enc.attention_mask.squeeze(0)

def build_dpo_dataset(ds):
    examples = []
    for ex in ds:
        chosen_ids, chosen_mask = tokenize_pair(ex["prompt"], ex["chosen"])
        rejected_ids, rejected_mask = tokenize_pair(ex["prompt"], ex["rejected"])
        examples.append({
            "chosen_input_ids": chosen_ids,
            "chosen_attention_mask": chosen_mask,
            "rejected_input_ids": rejected_ids,
            "rejected_attention_mask": rejected_mask
        })
    return examples

train_examples = build_dpo_dataset(train_pref)
val_examples = build_dpo_dataset(val_pref)

In [27]:
class DPODataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

train_loader = DataLoader(DPODataset(train_examples), batch_size=4, shuffle=True)
val_loader = DataLoader(DPODataset(val_examples), batch_size=4, shuffle=False)

In [28]:
# %%
# --- DPO training setup ---
beta = 0.1
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=5e-5)

num_epochs = 3
total_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

In [29]:
# Helper: log likelihood per sequence
def compute_log_prob(model, input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
    return -outputs.loss

# %%
# --- DPO Training Loop ---
for epoch in range(num_epochs):
    policy_model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        chosen_ids = batch["chosen_input_ids"].to(device)
        chosen_mask = batch["chosen_attention_mask"].to(device)
        rejected_ids = batch["rejected_input_ids"].to(device)
        rejected_mask = batch["rejected_attention_mask"].to(device)

        # Log likelihoods
        logp_chosen = compute_log_prob(policy_model, chosen_ids, chosen_mask)
        logp_rejected = compute_log_prob(policy_model, rejected_ids, rejected_mask)
        logp_chosen_ref = compute_log_prob(ref_model, chosen_ids, chosen_mask)
        logp_rejected_ref = compute_log_prob(ref_model, rejected_ids, rejected_mask)

        # DPO loss
        logits = beta * ((logp_chosen - logp_chosen_ref) - (logp_rejected - logp_rejected_ref))
        target = torch.ones_like(logits)
        loss = loss_fn(logits, target)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} — DPO Loss: {total_loss / len(train_loader):.4f}")

# %%
# Save the DPO-fine-tuned model
policy_model.save_pretrained("dpo_anthropic_model")
tokenizer.save_pretrained("dpo_anthropic_model")


Epoch 0 — DPO Loss: 0.0803
Epoch 1 — DPO Loss: 0.0001
Epoch 2 — DPO Loss: 0.0001


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


('dpo_anthropic_model/tokenizer_config.json',
 'dpo_anthropic_model/tokenizer.json')

In [34]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load your DPO-fine-tuned model
model_path = "dpo_anthropic_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Example prompt
prompt = "Bruh you know what time it is?"  # You can change this to any prompt you'd like

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=MAX_LENGTH,
        do_sample=True,      # sample outputs instead of greedy
        top_p=0.9,           # nucleus sampling
        temperature=0.7,
        num_return_sequences=1
    )

# Decode and print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5700.04it/s]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bruh you know what time it is? I'm gonna go get my cup of coffee." I said, "Hello?" She said, "Hello?" I said, Hello HelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHelloHel